# Chat 01 — Validación y consolidación de datos DANE

**Tesis:** Protocolo de generación de datos sintéticos de transacciones financieras para Colombia — Sara J. Cárdenas
**Objetivo del notebook:** reconstruir de forma **reproducible y con factores de expansión (FEX)** las cifras
demográficas que el informe de avance reporta, corrigiendo los hallazgos **H1–H8** del plan maestro.

Principios (criterios de aceptación):
- **Cero cifras hardcodeadas:** toda figura sale de un `value_counts`/`groupby`/media ponderada real.
- **FEX siempre:** GEIH → `FEX_C18`; ENPH → `FEX_C`. Se reportan versiones *sin ponderar* (para reproducir el informe) y *ponderadas* (corregidas).
- **Determinista:** semilla fija, rutas relativas, sin dependencia de estado externo.
- **Unidad de análisis explícita** en cada cifra (persona ocupada / hogar).
- **Urbano/rural declarados por separado** donde la fuente lo permite.

> Todas las cifras que aparecen en la tabla de reconciliación final se calculan en este notebook; ninguna se escribe a mano.

## Estructura de datos esperada

Coloca los microdatos DANE bajo `./data/` con este layout (rutas relativas):

```
data/
├── enph/
│   ├── viviendas_y_hogares.csv                 # ENPH: estrato P8520S1A1, medio pago NC2R_CC_P6
│   ├── gastos_menos_frecuentes_medio_pago.csv  # ENPH: batería P438* (29 columnas)
│   └── gastos_diarios_urbanos.csv              # ENPH: NH_CGDU_* (forma de adquisición, NO medio de pago)
└── geih/
    └── 2025-01/                                # una carpeta AAAA-MM por mes; 2026-02 se excluye
        ├── ocupados.csv                        # GEIH: P6440 contrato, P6500 ingreso, FEX_C18
        └── datos_hogar_vivienda.csv            # GEIH: estrato P4030S1A1, FEX_C18
```

Para reproducir con los **12 meses de 2025**, crea `data/geih/2025-02/ … 2025-12/` con los mismos dos archivos; el notebook autodetecta los meses y ajusta `FEX_C18 / n_meses`. Todos los CSV usan separador `;`. La E/S es robusta a `utf-8-sig`/`latin-1` y a los tres formatos de FEX de DANE.

## 0. Configuración: semilla, rutas relativas, mapeos de códigos

In [1]:
from __future__ import annotations
import os, glob, random, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd

# ---- Determinismo ----
SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ---- Rutas relativas (con fallback para ejecución en el sandbox) ----
def resolve_data_dir() -> Path:
    for cand in [Path("data"), Path("../data"), Path("/mnt/user-data/uploads")]:
        if cand.exists():
            return cand
    raise FileNotFoundError("No se encontró el directorio de datos. Coloca los CSV en ./data/")

DATA = resolve_data_dir()
ENPH = DATA / "enph"
GEIH = DATA / "geih"
print("DATA_DIR =", DATA.resolve())

# ---- Diccionario de códigos (Cuadernillo 3 ENPH + codebooks DANE) ----
# Medio de pago (P438* y NC2R_CC_P6):
MEDIO_PAGO = {1: "Tarjeta débito", 2: "Tarjeta crédito", 3: "Efectivo",
              4: "Bonos", 5: "Cheques", 6: "Otros"}
COD_EFECTIVO = 3
# GEIH P6440 (¿tiene contrato?) — verificado contra los datos: 2 = verbal, 1 = escrito
CONTRATO = {1: "Escrito", 2: "Verbal"}
COD_VERBAL = 2
# CLASE (GEIH): 1 = cabecera (urbano), 2 = resto (rural)
CLASE = {1: "Urbano (cabecera)", 2: "Rural (resto)"}
ESTRATOS_VALIDOS = [1, 2, 3, 4, 5, 6]  # 0/8/9/blanco = no aplica / no sabe / sin dato

# Contenedor de resultados para la tabla de reconciliación (nada hardcodeado)
REC = []  # cada item: dict(indicador, informe, recalc, coincide, comentario)
def registrar(indicador, informe, recalc, coincide, comentario):
    REC.append(dict(indicador=indicador, cifra_informe=informe,
                    cifra_recalculada=recalc, coincide=coincide, comentario=comentario))

DATA_DIR = /home/claude/chat01/data


## 1. Utilidades: E/S robusta y estadística ponderada

In [2]:
def leer_csv(path: Path, usecols=None) -> pd.DataFrame:
    """Lector robusto para microdatos DANE: separador ';', encoding utf-8-sig o latin-1,
    normaliza nombres de columna (quita BOM/espacios)."""
    last = None
    for enc in ("utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(path, sep=";", dtype=str, encoding=enc,
                             usecols=(lambda c: c.strip().lstrip("\ufeff") in usecols) if usecols else None)
            df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
            return df
        except (UnicodeDecodeError, Exception) as e:  # noqa
            last = e
    raise last

def num(s: pd.Series) -> pd.Series:
    """A numérico tolerando decimal por coma (FEX ENPH) y espacios."""
    return pd.to_numeric(s.astype(str).str.strip().str.replace(",", ".", regex=False),
                         errors="coerce")

def fex_num(s: pd.Series) -> pd.Series:
    # Parser robusto de factores de expansión con 3 formatos de export DANE:
    #  - ENPH: coma decimal ("39,3686")  -> quitar puntos de miles, coma->punto
    #  - GEIH Ocupados: punto decimal ("2184.7522833")
    #  - GEIH Datos_hogar: puntos de miles con 7 decimales implícitos ("21.847.522.833")
    t = s.astype(str).str.strip()
    out = pd.Series(np.nan, index=t.index, dtype="float64")
    coma = t.str.contains(",", na=False)
    out[coma] = pd.to_numeric(t[coma].str.replace(".", "", regex=False)
                                      .str.replace(",", ".", regex=False), errors="coerce")
    resto = ~coma
    ndots = t.str.count(r"\.")
    mang = resto & (ndots >= 2)   # puntos de miles -> 7 decimales implícitos
    out[mang] = pd.to_numeric(t[mang].str.replace(".", "", regex=False), errors="coerce") / 1e7
    norm = resto & (ndots <= 1)   # decimal normal o entero
    out[norm] = pd.to_numeric(t[norm], errors="coerce")
    return out

def wshare(valores: pd.Series, pesos: pd.Series, cond) -> float:
    """Proporción ponderada (%) de las filas que cumplen `cond` sobre el total con peso."""
    v = valores; w = pesos.astype(float)
    m = w.notna()
    return 100.0 * w[m & cond(v)].sum() / w[m].sum()

def wvalue_counts(valores: pd.Series, pesos: pd.Series, normalize=True) -> pd.Series:
    df = pd.DataFrame({"v": valores, "w": pesos.astype(float)}).dropna(subset=["w"])
    g = df.groupby("v")["w"].sum().sort_index()
    return (100 * g / g.sum()) if normalize else g

def wmean(x: pd.Series, w: pd.Series) -> float:
    m = x.notna() & w.notna()
    return float(np.average(x[m], weights=w[m])) if m.any() else np.nan

def wquantile(x: pd.Series, w: pd.Series, q) -> np.ndarray:
    """Cuantiles ponderados por interpolación sobre la CDF acumulada de pesos."""
    m = x.notna() & w.notna()
    xs = np.asarray(x[m], float); ws = np.asarray(w[m], float)
    order = np.argsort(xs); xs, ws = xs[order], ws[order]
    cw = np.cumsum(ws) - 0.5 * ws
    cw /= ws.sum()
    return np.interp(np.atleast_1d(q), cw, xs)

print("Utilidades cargadas.")

Utilidades cargadas.


## 2. Carga GEIH — apilado multi-mes con FEX ajustado

**Regla (criterio de aceptación):** meses del mismo módulo → **apilar** con `FEX_C18` **dividido por el número de meses**
(así la expansión estima un *mes promedio* y no la suma de 12 poblaciones). Módulos distintos → **unir por llave**
`DIRECTORIO + SECUENCIA_P + HOGAR`.

**Alcance real (H4):** se detectan automáticamente los meses disponibles bajo `data/geih/`. **Febrero 2026 se excluye
explícitamente.** Si solo hay enero, *todas las cifras GEIH son de enero 2025* — el informe dice "todos los meses",
lo que constituye el hallazgo **H4**.

In [3]:
def cargar_geih_modulo(patron_archivo: str, usecols) -> pd.DataFrame:
    """Apila un módulo GEIH sobre todas las carpetas de mes (data/geih/AAAA-MM/),
    excluye 2026-02, y agrega FEX_C18_adj = FEX_C18 / n_meses."""
    carpetas = sorted([p for p in GEIH.glob("*") if p.is_dir()])
    # Excluir febrero 2026 explícitamente
    carpetas = [p for p in carpetas if p.name != "2026-02"]
    if not carpetas:  # fallback: archivos sueltos directamente en data/ (solo enero)
        carpetas = [GEIH] if GEIH.exists() else []
    frames, meses = [], []
    for carp in carpetas:
        hits = list(carp.glob(patron_archivo))
        if not hits:
            continue
        df = leer_csv(hits[0], usecols=usecols | {"FEX_C18", "MES", "CLASE"})
        etiqueta = carp.name if carp.name != GEIH.name else str(df.get("MES", pd.Series(["01"])).iloc[0]).zfill(2)
        df["_MES"] = etiqueta
        frames.append(df); meses.append(etiqueta)
    if not frames:
        raise FileNotFoundError(f"No se hallaron archivos {patron_archivo} bajo {GEIH}")
    out = pd.concat(frames, ignore_index=True)
    n_meses = out["_MES"].nunique()
    out["FEX_C18"] = fex_num(out["FEX_C18"])
    out["FEX_C18_adj"] = out["FEX_C18"] / n_meses
    print(f"[{patron_archivo}] meses cargados = {sorted(set(meses))}  (n={n_meses})  filas={len(out):,}")
    return out

N_MESES_GEIH = None  # se fija abajo

ocu = cargar_geih_modulo("*ocupados*.csv", usecols={"DIRECTORIO","SECUENCIA_P","HOGAR","ORDEN","P6440","P6500"})
hog = cargar_geih_modulo("*hogar*.csv",    usecols={"DIRECTORIO","SECUENCIA_P","HOGAR","P4030S1A1"})
N_MESES_GEIH = ocu["_MES"].nunique()

# Tipar variables de interés
ocu["P6440"] = num(ocu["P6440"])
ocu["P6500"] = num(ocu["P6500"])
ocu["CLASE"] = num(ocu["CLASE"])
hog["P4030S1A1"] = num(hog["P4030S1A1"]).astype("Int64")

if N_MESES_GEIH == 1:
    print("\n⚠️  H4: solo hay 1 mes de GEIH (enero 2025). Las cifras GEIH son ENERO, no 'todos los meses'.")

[*ocupados*.csv] meses cargados = ['2025-01']  (n=1)  filas=28,317


[*hogar*.csv] meses cargados = ['2025-01']  (n=1)  filas=24,109

⚠️  H4: solo hay 1 mes de GEIH (enero 2025). Las cifras GEIH son ENERO, no 'todos los meses'.


## 3. ENPH — carga (estrato, medio de pago diario, P438) con FEX

In [4]:
# --- Viviendas y hogares (estrato ENPH P8520S1A1 + medio de pago diario NC2R_CC_P6) ---
viv = leer_csv(ENPH / "viviendas_y_hogares.csv",
               usecols={"DIRECTORIO","SECUENCIA_ENCUESTA","SECUENCIA_P","ORDEN",
                        "P8520S1A1","NC2R_CC_P6","DOMINIO","FEX_C"})
viv["P8520S1A1"] = num(viv["P8520S1A1"]).astype("Int64")
viv["NC2R_CC_P6"] = num(viv["NC2R_CC_P6"])   # blancos -> NaN = 'sin dato'
viv["FEX_C"] = fex_num(viv["FEX_C"])
print("ENPH Viviendas y hogares:", f"{len(viv):,} hogares")

# Clasificación urbano/rural ENPH a partir de DOMINIO
viv["_ZONA"] = np.where(viv["DOMINIO"].astype(str).str.upper()
                        .str.contains("RURAL|CENTRO POBLADO", na=False),
                        "Rural", "Urbano")
print(viv["_ZONA"].value_counts(dropna=False).to_dict())

# --- Gastos menos frecuentes - Medio de pago (batería P438*) ---
h438 = leer_csv(ENPH / "gastos_menos_frecuentes_medio_pago.csv")
p438_cols = [c for c in h438.columns if c.startswith("P438")]
h438["FEX_C"] = fex_num(h438["FEX_C"])
print(f"\nENPH Gastos menos frecuentes: {len(h438):,} hogares, {len(p438_cols)} columnas P438* de medio de pago")
print("P438*:", p438_cols)

ENPH Viviendas y hogares: 87,201 hogares
{'Urbano': 81438, 'Rural': 5763}



ENPH Gastos menos frecuentes: 87,201 hogares, 29 columnas P438* de medio de pago
P438*: ['P438A11', 'P438B11', 'P438B12', 'P438C11', 'P438C12', 'P438D11', 'P438D12', 'P438D13', 'P438D14', 'P438D15', 'P438D16', 'P438E11', 'P438E12', 'P438F11', 'P438F12', 'P438F13', 'P438G11', 'P438G12', 'P438G13', 'P438G14', 'P438H11', 'P438H12', 'P438I11', 'P438I12', 'P438J11', 'P438J12', 'P438J13', 'P438L11', 'P438L12']


### 3.1 Verificación: `Gastos_diarios_Urbanos` **no** tiene variable de medio de pago

El informe atribuye el 73.1% de efectivo a los "gastos diarios", pero ese archivo (`NH_CGDU_*`) solo tiene
**"Forma de Adquisición"** (compra/autoconsumo/etc.), **no medio de pago**. La cifra de 73.1% proviene de
`NC2R_CC_P6` (módulo Viviendas y hogares), no de este archivo. Se documenta y se registra el tamaño.

In [5]:
gdiag = leer_csv(ENPH / "gastos_diarios_urbanos.csv", usecols={"DIRECTORIO","FEX_C"})
tiene_medio_pago = any("NC2R_CC_P6" == c or "medio_de_pago" in c.lower() for c in
                       leer_csv(ENPH / "gastos_diarios_urbanos.csv").columns)
n_gd = len(gdiag)
print(f"Gastos diarios urbanos: {n_gd:,} registros (nivel transacción/artículo).")
print("¿Contiene variable de medio de pago?:", tiene_medio_pago)
if n_gd == 1_048_575:
    print("⚠️  RIESGO: exactamente 1.048.575 filas = límite de filas de Excel (2^20 - 1). "
          "Posible truncamiento del archivo en origen. Verificar export DANE.")

Gastos diarios urbanos: 1,048,575 registros (nivel transacción/artículo).
¿Contiene variable de medio de pago?: False
⚠️  RIESGO: exactamente 1.048.575 filas = límite de filas de Excel (2^20 - 1). Posible truncamiento del archivo en origen. Verificar export DANE.


## 4. ENPH — Estrato socioeconómico (H3, H6)

- **H6:** el informe mezcla **dos fuentes/años** para el estrato: la *distribución* viene de ENPH 2016-17
  (`P8520S1A1`) y el *ingreso por estrato* de GEIH 2025 (`P4030S1A1`). Se calculan ambas distribuciones
  para hacerlo explícito.
- **H3:** el informe usa proporciones **crudas**. Se reporta cruda (reproduce) y **ponderada por FEX** (corregida).
- **Unidad:** hogar.

In [6]:
# Distribución cruda tal como el informe (base = TODOS los hogares, códigos 1-6 mostrados)
base_total = len(viv)
cruda_informe = (viv["P8520S1A1"].value_counts().reindex(ESTRATOS_VALIDOS) / base_total * 100)

# Ponderada por FEX, base = hogares con estrato válido (corregida)
val = viv[viv["P8520S1A1"].isin(ESTRATOS_VALIDOS)]
pond = wvalue_counts(val["P8520S1A1"], val["FEX_C"]).reindex(ESTRATOS_VALIDOS)

estrato_enph = pd.DataFrame({
    "% cruda (informe, base=todos)": cruda_informe.round(2),
    "% ponderada FEX (corregida)": pond.round(2),
})
estrato_enph.index.name = "Estrato (ENPH P8520S1A1)"
display(estrato_enph)

# También la distribución GEIH (P4030S1A1) para evidenciar H6
valg = hog[hog["P4030S1A1"].isin(ESTRATOS_VALIDOS)]
estrato_geih = wvalue_counts(valg["P4030S1A1"], valg["FEX_C18_adj"]).reindex(ESTRATOS_VALIDOS).round(2)
print("\nEstrato GEIH 2025 (P4030S1A1), ponderado — para contraste con ENPH (H6):")
print(estrato_geih.to_dict())

# Registrar reconciliación (E1, E2, E3 y suma E4-6)
for e, ref in [(1, "36.3%"), (2, "34.5%"), (3, "20.2%")]:
    rc = cruda_informe[e]
    ok = "sí" if abs(rc - float(ref[:-1])) < 0.15 else "no"
    registrar(f"Estrato ENPH E{e} (%)", ref, f"{rc:.2f}% (cruda) / {pond[e]:.2f}% (FEX)",
              ok, "Cruda reproduce el informe; ponderada FEX difiere. Fuente ENPH 2016-17, no GEIH (H6).")
suma_46_inf = 7.5
suma_46_rc = float(cruda_informe.reindex([4,5,6]).sum())
registrar("Estrato ENPH E4+E5+E6 (%)", f"{suma_46_inf}%",
          f"{suma_46_rc:.2f}% (cruda) / {pond.reindex([4,5,6]).sum():.2f}% (FEX)",
          "sí" if abs(suma_46_rc-suma_46_inf) < 0.15 else "no",
          "Reproduce; el informe atribuye la distribución a GEIH pero proviene de ENPH (H6).")

,"% cruda (informe, base=todos)",% ponderada FEX (corregida)
Estrato (ENPH P8520S1A1),,
1,36.2800,33.8000
2,34.5100,36.3300
3,20.1700,22.5100
4,5.0000,4.6100
5,1.6800,1.9400
6,0.8000,0.8100



Estrato GEIH 2025 (P4030S1A1), ponderado — para contraste con ENPH (H6):
{1: 41.81, 2: 31.02, 3: 18.97, 4: 5.17, 5: 2.18, 6: 0.86}


## 5. ENPH — Medio de pago en gastos diarios `NC2R_CC_P6` (73.1%)

Este cálculo el informe **sí lo hace en vivo** (nota del plan). Se verifica.
Escala Cuadernillo 3: 1 débito · 2 crédito · **3 efectivo** · 4 bonos · 5 cheques · 6 otros; **blanco = sin dato**.
**Unidad:** hogar.

In [7]:
n_viv = len(viv)
# Cruda como el informe: base = todos los hogares; 'sin dato' = blancos (NaN)
efvo_cruda = 100 * (viv["NC2R_CC_P6"] == COD_EFECTIVO).sum() / n_viv
sindato_cruda = 100 * viv["NC2R_CC_P6"].isna().sum() / n_viv
deb_cruda = 100 * (viv["NC2R_CC_P6"] == 1).sum() / n_viv
cred_cruda = 100 * (viv["NC2R_CC_P6"] == 2).sum() / n_viv
# Ponderada por FEX (misma base = todos los hogares con FEX)
efvo_pond = wshare(viv["NC2R_CC_P6"], viv["FEX_C"], lambda v: v == COD_EFECTIVO)

print(f"Efectivo (cruda)   : {efvo_cruda:.2f}%   | ponderada FEX: {efvo_pond:.2f}%")
print(f"Sin dato (blanco)  : {sindato_cruda:.2f}%")
print(f"Débito {deb_cruda:.2f}%  ·  Crédito {cred_cruda:.2f}%")

registrar("Efectivo gastos diarios NC2R_CC_P6 (%)", "73.1%",
          f"{efvo_cruda:.2f}% (cruda) / {efvo_pond:.2f}% (FEX)",
          "sí" if abs(efvo_cruda-73.1) < 0.2 else "no",
          "Reproduce sin ponderar; base = todos los hogares, blanco=sin dato (25.5%). Ponderada FEX difiere.")

Efectivo (cruda)   : 73.06%   | ponderada FEX: 69.59%
Sin dato (blanco)  : 25.47%
Débito 0.31%  ·  Crédito 0.20%


## 6. ENPH — P438 "gastos menos frecuentes / medio de pago" (97.6%) — **H1, H2**

**H1/H2:** en el notebook original la cifra estaba **hardcodeada** (`pago_cols = []  # TODO`).
Aquí se **deriva**: se apilan las **29 columnas** de medio de pago de la batería P438 (una por subcapítulo de gasto:
alimentos, cuidado personal, vivienda, vestir, educación, salud, electrodomésticos, transporte, vehículo,
recreación…), se conservan solo valores válidos 1–6, y se calcula la proporción de efectivo (código 3).

**Riesgo metodológico (clave):** esta es una proporción de **"instancias de medio de pago por subcapítulo"**,
ponderada por la frecuencia con que cada hogar tuvo cada tipo de gasto — **no** es una proporción de transacciones ni
de pesos. Un hogar con más categorías de gasto aporta más observaciones. **Unidad ambigua**; declararla al citar.

In [8]:
# Confirmar que las 29 columnas son de la escala de medio de pago (todas 1-6)
vals_por_col = {c: sorted(pd.to_numeric(h438[c], errors="coerce").dropna().unique().tolist())
                for c in p438_cols}
ok_escala = all(set(v) <= {1,2,3,4,5,6} for v in vals_por_col.values())
print("Todas las columnas P438* usan la escala 1-6 de medio de pago:", ok_escala)

# Apilar (pooled) todas las respuestas válidas + su FEX de hogar
val_list, fex_list = [], []
for c in p438_cols:
    v = pd.to_numeric(h438[c], errors="coerce")
    m = v.isin([1,2,3,4,5,6])
    val_list.append(v[m]); fex_list.append(h438["FEX_C"][m])
Vp = pd.concat(val_list, ignore_index=True)
Fp = pd.concat(fex_list, ignore_index=True)
print(f"Instancias válidas de medio de pago (pooled sobre {len(p438_cols)} subcapítulos): {len(Vp):,}")

ef_p438_cruda = 100 * (Vp == COD_EFECTIVO).mean()
ef_p438_pond  = wshare(Vp, Fp, lambda v: v == COD_EFECTIVO)
print(f"Efectivo P438 (cruda): {ef_p438_cruda:.2f}%  |  ponderada FEX: {ef_p438_pond:.2f}%")

dist_p438 = wvalue_counts(Vp, Fp).rename(index=MEDIO_PAGO).round(2)
print("Distribución ponderada P438:", dist_p438.to_dict())

registrar("Efectivo P438 gastos menos frec. (%)", "97.6%",
          f"{ef_p438_cruda:.2f}% (cruda) / {ef_p438_pond:.2f}% (FEX)",
          "sí" if abs(ef_p438_cruda-97.6) < 0.2 else "no",
          "Reproduce la cifra hardcodeada al apilar las 29 columnas de medio de pago sin ponderar. "
          "Es proporción de instancias por subcapítulo, no de transacciones; ponderada FEX ≈ 96%.")

Todas las columnas P438* usan la escala 1-6 de medio de pago: True


Instancias válidas de medio de pago (pooled sobre 29 subcapítulos): 631,916
Efectivo P438 (cruda): 97.56%  |  ponderada FEX: 96.15%
Distribución ponderada P438: {'Tarjeta débito': 1.74, 'Tarjeta crédito': 1.73, 'Efectivo': 96.15, 'Bonos': 0.08, 'Cheques': 0.02, 'Otros': 0.29}


## 7. GEIH — Ocupados: conteo válido (14.975), contrato verbal (40.6%) — **H4, H8**

- **14.975** = conteo **sin ponderar** de ocupados con `P6500 > 0` en **enero 2025** (H4: no "todos los meses").
- **40.6%** contrato verbal = `P6440 == 2`, **sin ponderar** (unidad: **persona ocupada**).
- Se añade la versión **ponderada** (población) y el desglose **urbano/rural** (`CLASE`).

In [9]:
# Conteo de ocupados con ingreso positivo (sin ponderar) — reproduce 14.975 en enero
ocu_ing = ocu[ocu["P6500"] > 0].copy()
n_ing = len(ocu_ing)
print(f"Ocupados con P6500>0 (sin ponderar): {n_ing:,}  [meses: {sorted(ocu['_MES'].unique())}]")
pob_ocupados = ocu["FEX_C18_adj"].sum()
print(f"Población de ocupados (FEX ajustado por {N_MESES_GEIH} mes/es): {pob_ocupados:,.0f}")

registrar("Ocupados con ingreso (conteo)", "14.975",
          f"{n_ing:,} (enero 2025, sin ponderar)",
          "sí" if n_ing == 14975 else "no",
          f"Reproduce, pero es SOLO enero (H4). Población ponderada ≈ {pob_ocupados:,.0f}.")

# Contrato verbal (P6440==2). Unidad: persona ocupada.
verbal_cruda = 100 * (ocu["P6440"] == COD_VERBAL).sum() / ocu["P6440"].isin([1,2]).sum()
verbal_pond  = wshare(ocu["P6440"], ocu["FEX_C18_adj"], lambda v: v == COD_VERBAL)
print(f"\nContrato verbal (cruda): {verbal_cruda:.2f}%  |  ponderada FEX: {verbal_pond:.2f}%")

# Desglose urbano/rural (ponderado)
for cl, etq in CLASE.items():
    sub = ocu[ocu["CLASE"] == cl]
    v = wshare(sub["P6440"], sub["FEX_C18_adj"], lambda x: x == COD_VERBAL)
    print(f"  - {etq}: verbal (FEX) = {v:.2f}%")

registrar("Contrato verbal P6440 (%)", "40.6%",
          f"{verbal_cruda:.2f}% (cruda) / {verbal_pond:.2f}% (FEX)",
          "sí" if abs(verbal_cruda-40.6) < 0.2 else "no",
          "Reproduce sin ponderar (enero). Unidad: persona ocupada. Ponderada FEX difiere.")

Ocupados con P6500>0 (sin ponderar): 14,975  [meses: ['2025-01']]
Población de ocupados (FEX ajustado por 1 mes/es): 22,902,835

Contrato verbal (cruda): 40.60%  |  ponderada FEX: 40.42%
  - Urbano (cabecera): verbal (FEX) = 35.52%
  - Rural (resto): verbal (FEX) = 58.85%


## 8. GEIH — Ingreso por quintil (H7) y por estrato (H6, H8)

- **H7:** el informe usó `pd.qcut` sobre `P6500` **crudo** (sin ponderar, con empates) → rangos irregulares/solapados.
  Aquí se calculan **fronteras de quintil ponderadas** por FEX.
- **H8:** `P6500` es ingreso **laboral del ocupado**, no del hogar. La cifra "por estrato" es el **ingreso laboral medio
  de las personas ocupadas** que viven en cada estrato. **Unidad: persona ocupada** (se declara).
- **Cruce de módulos** por llave `DIRECTORIO + SECUENCIA_P + HOGAR`.

In [10]:
x = ocu_ing["P6500"].astype(float)
w = ocu_ing["FEX_C18_adj"].astype(float)

# --- Quintiles ponderados (H7) ---
cortes = wquantile(x, w, [0.2, 0.4, 0.6, 0.8])
bordes = np.concatenate([[x.min()], cortes, [x.max()]])
q_idx = np.clip(np.digitize(x, cortes, right=False), 0, 4)
tab_q = (pd.DataFrame({"P6500": x, "w": w, "Q": q_idx})
         .groupby("Q").apply(lambda g: pd.Series({
             "Ingreso medio (COP)": np.average(g["P6500"], weights=g["w"]),
             "Límite inferior": g["P6500"].min(),
             "Límite superior": g["P6500"].max(),
         })))
tab_q.index = [f"Q{i+1}" for i in tab_q.index]
print("Ingreso por quintil — PONDERADO (corrige H7):")
display(tab_q.round(0))
print("Fronteras de quintil (ponderadas):", np.round(cortes, 0).tolist())

# --- Ingreso por estrato (H6/H8): unir ocupados x datos del hogar ---
llave = ["DIRECTORIO", "SECUENCIA_P", "HOGAR"]
oj = ocu_ing.merge(hog[llave + ["P4030S1A1"]].drop_duplicates(llave), on=llave, how="left")
cob = oj["P4030S1A1"].isin(ESTRATOS_VALIDOS).mean() * 100
print(f"\nCobertura del cruce (ocupados con estrato válido): {cob:.1f}%")

tab_e = (oj[oj["P4030S1A1"].isin(ESTRATOS_VALIDOS)]
         .groupby("P4030S1A1")
         .apply(lambda g: wmean(g["P6500"], g["FEX_C18_adj"])))
tab_e.index.name = "Estrato"; tab_e.name = "Ingreso laboral medio ocupado (COP)"
print("Ingreso por estrato — PONDERADO (unidad: persona ocupada; H8):")
display(tab_e.round(0).to_frame())

# Reconciliación ingreso por estrato vs informe (Tabla III)
ref_estrato = {1:1405582, 2:1732948, 3:2132129, 4:3740578, 5:5746814, 6:6648988}
for e in ESTRATOS_VALIDOS:
    rc = float(tab_e.get(e, np.nan))
    dif = 100*abs(rc-ref_estrato[e])/ref_estrato[e]
    registrar(f"Ingreso medio estrato {e} (COP)", f"${ref_estrato[e]:,.0f}",
              f"${rc:,.0f}", "no",
              f"Difiere ~{dif:.0f}%: informe usó ingreso crudo sin FEX; aquí ponderado. Unidad: ocupado (H8).")

ref_q = {1:750291, 2:1299709, 3:1440520, 4:1927110, 5:4898257}
for i,qn in enumerate([f"Q{k}" for k in range(1,6)]):
    rc = float(tab_q.loc[qn, "Ingreso medio (COP)"])
    registrar(f"Ingreso medio {qn} (COP)", f"${ref_q[i+1]:,.0f}", f"${rc:,.0f}", "no",
              "Fronteras/medias ponderadas; informe usó qcut crudo con empates (H7).")

Ingreso por quintil — PONDERADO (corrige H7):


,Ingreso medio (COP),Límite inferior,Límite superior
Q1,"656,266.0000","10,000.0000","1,195,000.0000"
Q2,"1,205,537.0000","1,200,000.0000","1,292,000.0000"
Q3,"1,336,730.0000","1,300,000.0000","1,495,000.0000"
Q4,"1,802,379.0000","1,500,000.0000","2,465,764.0000"
Q5,"5,340,562.0000","2,500,000.0000","42,000,000.0000"


Fronteras de quintil (ponderadas): [1200000.0, 1300000.0, 1500000.0, 2500000.0]

Cobertura del cruce (ocupados con estrato válido): 98.6%
Ingreso por estrato — PONDERADO (unidad: persona ocupada; H8):


,Ingreso laboral medio ocupado (COP)
Estrato,
1,"1,360,291.0000"
2,"1,722,145.0000"
3,"2,138,842.0000"
4,"4,483,338.0000"
5,"7,520,805.0000"
6,"8,867,473.0000"


## 9. Tabla de reconciliación — cifra informe → recalculada → ¿coincide?

In [11]:
rec = pd.DataFrame(REC, columns=["indicador","cifra_informe","cifra_recalculada","coincide","comentario"])
display(rec)

# Exportar nota_hallazgos.md desde valores REALES (sin hardcode)
def md_table(df):
    heads = ["Indicador","Cifra informe","Cifra recalculada","¿Coincide?","Comentario"]
    out = ["| " + " | ".join(heads) + " |", "|" + "|".join(["---"]*len(heads)) + "|"]
    for _,r in df.iterrows():
        out.append("| " + " | ".join(str(r[c]).replace("\n"," ") for c in
                   ["indicador","cifra_informe","cifra_recalculada","coincide","comentario"]) + " |")
    return "\n".join(out)

n_meses_txt = ", ".join(sorted(ocu["_MES"].unique()))
nota = f"""# Nota de hallazgos — Validación de datos DANE (Chat 01)

**Autora:** Sara J. Cárdenas · **Notebook:** `01_validacion_datos.ipynb` · **Semilla:** {SEED}
**Alcance de datos cargados:** GEIH meses = [{n_meses_txt}] (n={N_MESES_GEIH}); ENPH 2016-2017.
Todas las cifras se derivan de `value_counts`/`groupby`/medias ponderadas en el notebook; ninguna está escrita a mano.

## Tabla de reconciliación

{md_table(rec)}

## Lectura de los hallazgos H1–H8

- **H1 (P438 97.6% hardcodeado):** la cifra **se reproduce** al apilar las {len(p438_cols)} columnas de medio de pago
  de la batería P438 y tomar la proporción de efectivo (código 3) **sin ponderar** ({ef_p438_cruda:.2f}%).
  Estaba hardcodeada; ahora se deriva. Ponderada por FEX ≈ {ef_p438_pond:.1f}%. **Riesgo:** es proporción de
  *instancias de pago por subcapítulo*, no de transacciones ni de pesos → declarar la unidad al citarla.
- **H2 (cifras a mano):** todas las figuras salen ahora de conteos/agrupaciones reales.
- **H3 (sin FEX):** cada indicador se reporta *crudo* (reproduce el informe) y *ponderado por FEX* (corregido).
  Las versiones ponderadas difieren de las crudas → las cifras poblacionales del informe deben migrar a FEX.
- **H4 (GEIH solo enero):** confirmado — hay {N_MESES_GEIH} mes(es). El conteo de 14.975 ocupados y el 40.6% verbal
  son de **enero 2025**, no "todos los meses". El notebook apila automáticamente los meses que se agreguen a
  `data/geih/AAAA-MM/` (excluyendo 2026-02) y ajusta FEX por número de meses.
- **H5 (código truncado):** el notebook corre de punta a punta sin celdas incompletas.
- **H6 (dos fuentes de estrato):** explícito — la *distribución* de estrato es ENPH 2016-17 (`P8520S1A1`) y el
  *ingreso por estrato* es GEIH 2025 (`P4030S1A1`); son encuestas y años distintos. Se muestran ambas distribuciones.
- **H7 (quintiles con qcut crudo):** se recalculan **fronteras y medias ponderadas** por FEX; difieren de la Tabla II.
- **H8 (unidad de ingreso):** `P6500` es ingreso **laboral del ocupado**, no del hogar. Las cifras por estrato/quintil
  se declaran como **ingreso laboral medio por persona ocupada**.

## Riesgos metodológicos adicionales
- `Gastos_diarios_Urbanos.csv` tiene {n_gd:,} filas. Si es exactamente 1.048.575 (= 2^20−1) hay **posible truncamiento**
  al límite de Excel; verificar el export original del DANE antes de usarlo en calibración.
- El 73.1% (diario), 97.6% (menos frecuente) y 77.8% (Banrep) son **universos distintos** (hogar vs. instancias por
  subcapítulo vs. valor de transacciones). No son intercambiables: declarar cuál calibra cada etapa del protocolo.
- Cruce ocupados × hogar por `DIRECTORIO+SECUENCIA_P+HOGAR`: cobertura de estrato ≈ {cob:.1f}% (documentar la pérdida).
"""

from pathlib import Path
Path("nota_hallazgos.md").write_text(nota, encoding="utf-8")
print("nota_hallazgos.md escrito (", len(nota), "caracteres ).")

,indicador,cifra_informe,cifra_recalculada,coincide,comentario
0,Estrato ENPH E1 (%),36.3%,36.28% (cruda) / 33.80% (FEX),sí,Cruda reproduce el informe; ponderada FEX difi...
1,Estrato ENPH E2 (%),34.5%,34.51% (cruda) / 36.33% (FEX),sí,Cruda reproduce el informe; ponderada FEX difi...
2,Estrato ENPH E3 (%),20.2%,20.17% (cruda) / 22.51% (FEX),sí,Cruda reproduce el informe; ponderada FEX difi...
3,Estrato ENPH E4+E5+E6 (%),7.5%,7.48% (cruda) / 7.36% (FEX),sí,Reproduce; el informe atribuye la distribución...
4,Efectivo gastos diarios NC2R_CC_P6 (%),73.1%,73.06% (cruda) / 69.59% (FEX),sí,Reproduce sin ponderar; base = todos los hogar...
5,Efectivo P438 gastos menos frec. (%),97.6%,97.56% (cruda) / 96.15% (FEX),sí,Reproduce la cifra hardcodeada al apilar las 2...
6,Ocupados con ingreso (conteo),14.975,"14,975 (enero 2025, sin ponderar)",sí,"Reproduce, pero es SOLO enero (H4). Población ..."
7,Contrato verbal P6440 (%),40.6%,40.60% (cruda) / 40.42% (FEX),sí,Reproduce sin ponderar (enero). Unidad: person...
8,Ingreso medio estrato 1 (COP),"$1,405,582","$1,360,291",no,Difiere ~3%: informe usó ingreso crudo sin FEX...
9,Ingreso medio estrato 2 (COP),"$1,732,948","$1,722,145",no,Difiere ~1%: informe usó ingreso crudo sin FEX...


nota_hallazgos.md escrito ( 5680 caracteres ).


## 10. Resumen de reproducibilidad

- **Sí reproducen** (sin ponderar, tal como el informe): P438 97.6%, NC2R_CC_P6 73.1%, estrato ENPH, 40.6% verbal, 14.975 ocupados.
- **Difieren al corregir** (FEX / ponderación / unidad): ingreso por estrato e ingreso por quintil, y todas las versiones ponderadas.
- Corregidos **H1–H8**; unidad de análisis y universo declarados en cada cifra; rutas relativas y semilla fija.